# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!


## Again - please note: pro-tip for using Colab:

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [3]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [4]:
# New capability - connect this Colab to your Google Drive
# See immediately below this for instructions to obtain denver_extract.mp3
# Place the file on your drive in a folder called llms, and call it denver_extract.mp3

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/denver_extract.mp3"

Mounted at /content/drive


# Download denver_extract.mp3

You can either use the same file as me, the extract from Denver city council minutes, or you can try your own..

If you want to use the same as me, then please download my extract here, and put this on your Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


In [5]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the file

audio_file = open(audio_filename, "rb")

# STEP 1: Transcribe Audio

## Option 1: Use Open Source for Transcription - Hugging Face Pipelines

In [6]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


 kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So, thank you. Thank you so much and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the 

In [8]:
open_source_transcription = transcription

## Option 2: Use OpenAI for Transcription

In [9]:
# Sign in to OpenAI using Secrets in Colab

# AUDIO_MODEL = "gpt-4o-mini-transcribe"

# openai_api_key = userdata.get('OPENAI_API_KEY')
# openai = OpenAI(api_key=openai_api_key)
# transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
# print(transcription)
# if openai is not available, downlaod whisper locally and run it
import whisper

model = whisper.load_model("base")   # tiny, base, small, medium, large

result = model.transcribe(audio_filename)
transcription = result["text"]
print(transcription)


 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous peoples day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th, please rise with the pledge of allegiance by Councilman Lopez. One makes me under God in the visible liberty 

In [10]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous peoples day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th, please rise with the pledge of allegiance by Councilman Lopez. One makes me under God in the visible liberty of the justice for all. All right. Thank you Councilman Lopez. Madam Secretary, roll call. Black, clerk, espinosa, here, Flynn, Gilmore, here, here, Cashman, here, can each, here, Lopez, new, here, or Teiga, here, Sussan, Mr. President, here, 11 present, 11 members present, we do have a quorum. Approval of the minutes, are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd, stand approved council announcement. So are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just want to do invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock PM. It will move along Broadway from 3rd to Alameda. It's going to be a fun family friendly event. Everyone's invited to come down. Where costume? There will be candy for the kids and there are teaky zombies and 29 herses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right. Thank you, Councilman Clark. We'll be there. All right. Presentations. Madam Secretary, we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening, proclamation 1127, an observance of the annual Indigenous People's Day in the City and County of Denver. Councilman Loplayes, would you please read it? Thank you, Mr. President. We're fried. Proclamation number 17, well let me just say this proclamation number 1127 series of 2017 and observance of the second annual Indigenous People's Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the Indigenous peoples have lived and flourished on the lands known as the Ametica since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes which include the southern Newt, the Yut Mountain, Yut tribes of Colorado and whereas the tribal homelands and seasonal encampments of the Arapoho and Shamb people along the banks of the Cherry Creek and South Plot River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes in the City and County of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas on October 3rd 2016 the City and County of Denver unanimously passed Council Bill 801 series of 2016 officially designating the second Monday of October of each year as Indigenous People's Day in Denver, Colorado. And whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to community, made to the community through Indigenous People's Knowledge, Science, Philosophy, Arts and Culture and through these contributions the City of Denver has developed and thrived. Whereas the Indigenous community especially Yut have made great efforts this year to draw attention to the contributions of Indigenous people including Confluence Week, drawing record of tennis to a National Indigenous Youth Leadership Conference leading conversations on inclusion with their peers and supporting increased Indigenous youth participation in science and engineering. Now therefore be it proclaimed by the Council of the City and County of Denver Section 1 that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history. Our past, present and future and continues to promote the education of the Denver community about these historic and contemporary contributions of Indigenous people. Section 2 at the City and County of Denver, Colorado does hereby observe October 9, 2017 as Indigenous People's Day. Section 3 at the clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation and that a copy be transmitted, excuse me, to the Denver American Indian Commission, the City and County of Denver School District number 1 and the Colorado Commission on Indian Affairs. Thank you Councilman Lopez, your motion to adopt. Mr. President, I move that proclamation number 1127, series of 2017 be adopted. All right, it has been moved and second, it comes with a council councilman Lopez. Thank you Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for the third time but as Indigenous people's day in Denver officially for the second time. It is, it's always awesome to be able to see not just this proclamation come through, come by my desk but to see so many different people from our community in our council chambers. It was a very beautiful piece of artwork that you presented to us earlier and it is exactly the spirit that we drafted this proclamation and this actual, the ordinance that created Indigenous People's Day when we sat down and wrote it and as a community we couldn't think of anything else to begin except for the confluence of the two rivers and those confluence of the two rivers created such a great city and we live in such an amazing city and we we're all proud of it and sometimes we and a lot of people from all over the country are out of the world are proud of it and sometimes a little too proud of it is telling them to go back home. But I'm kidding when I say that but the really nice thing about this is that we are celebrating Indigenous People's Day out of pride for who we are, who we are as a city and the contributions of Indigenous people to the city not out of spite, not out of a replacement of one culture over the other or out of contempt or disrespect. I think of a quote that Sisso Chavez made very very popular and it stuck with me for a very long time and anytime I have the opportunity I speak in front of children and especially children in our community that they often second guests themselves and where they're coming from, who they are and I always say that it's very important to be proud of who you're from and the quote that I use Sisso Chavez is pride in one's own culture does not require contempt or disrespect of another and that's very important. It's very important for us to recognize that no matter who we are where we come from in this society that your pride in your own culture doesn't require the contempt or disrespect of another. Amen, what a year to be for that to just sit on our shoulders for a while for us to think about. And so I wanted to just to thank you all, I think the commission, there's going to be a couple individuals that are going to come speak thank you for your art, your lovely artwork for us to see what's in your heart and what now has become a probably going to be a very important symbol for the community and also just for the work, the daily work every single day we still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners right in poverty without access to services right without access to sobriety or even housing or jobs and what a what a what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores right so we have a lot of work to do and these kind of proclamations in this day is not a day off it's a day on and then right and addressing those those those critical issues so I know that my colleagues are very supportive I'm going to ask you to support this proclamation as I know you always have done in the past I'm very proud of today oh and we made time magazine and newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous peoples day I wanted to make a point out of that thank you uh councilman Lopez and thank you for sponsoring this council martega Mr. President I want to ask that my name be added I don't think I could um add much more to what councilman Lopez has shared with us I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state I worked in the lieutenant governor's office when the commission on Indian affairs was created and had the benefit of being able to go down to the four corners for a peace treaty signing ceremony between the uts and the command cheese that had been sort of at odds with each other for about a hundred years and just being able to participate in that powwow was was pretty awesome so um and for those of you who continue to participate in the annual powwow it's it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture I mean to see that the dance continues to be carried on as well as as the native language from generation to generation is just so incredible because in so many cultures you know people have come here and um assimilated to the you know the norms here and they lose their language and and lose a lot of the culture and in the native community that that hasn't happened that has that you know commitment to just passing that on from generation to generation is is so important and so I'm happy to be a co-sponsor of this tonight thank you all right thank you councilman or take a councilwoman can you thank you very much and I also want to thank my colleague for bringing this forward and I I just wanted to to say a word to the artist about how beautiful and moving I thought this logo was and your description of it um and I think one of the things um that is clear is you know the words sometimes don't convey the power of imagery or music or the other pieces that make up culture and so I think the art is so important and when you talked about water I was also thinking about land and I guess I just wanted to say thank you um many of the native American peoples of Colorado have been at the forefront are actually nationally of defending some of the the um public lands that have been protected over the last few years that are under attack right now and their places that you uh the communities have fought to protect but that everyone gets to enjoy and so I just think that it's an example of where cultural preservation intersects with environmental protection with you know recreation and all of the other ways that that public lands are so important and so I think um I just wanted to say thank you for that because I think we have some very sacred places in our country that are at risk right now and um so as we celebrate I appreciate that there's still a piece of resistance in here and I think that um I just want to mention a solidarity and I mentioned a feeling of solidarity with that resistance so thank you and uh happy confluence week thank you uh councilman can each uh and see you know other comments I'll just say a couple and um in a time of such divisive um ugliness and just despicable behavior from our leadership um the reason I'm so supportive of indigenous peoples days because it means inclusivity it means respecting all respecting those who have been silenced um uh on purpose for a long time and whose history has not been told and so we celebrate inclusivity in the face of such um such evil types honestly

 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous peoples day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th, please rise with the pledge of allegiance by Councilman Lopez. One makes me under God in the visible liberty of the justice for all. All right. Thank you Councilman Lopez. Madam Secretary, roll call. Black, clerk, espinosa, here, Flynn, Gilmore, here, here, Cashman, here, can each, here, Lopez, new, here, or Teiga, here, Sussan, Mr. President, here, 11 present, 11 members present, we do have a quorum. Approval of the minutes, are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd, stand approved council announcement. So are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just want to do invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock PM. It will move along Broadway from 3rd to Alameda. It's going to be a fun family friendly event. Everyone's invited to come down. Where costume? There will be candy for the kids and there are teaky zombies and 29 herses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right. Thank you, Councilman Clark. We'll be there. All right. Presentations. Madam Secretary, we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening, proclamation 1127, an observance of the annual Indigenous People's Day in the City and County of Denver. Councilman Loplayes, would you please read it? Thank you, Mr. President. We're fried. Proclamation number 17, well let me just say this proclamation number 1127 series of 2017 and observance of the second annual Indigenous People's Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the Indigenous peoples have lived and flourished on the lands known as the Ametica since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes which include the southern Newt, the Yut Mountain, Yut tribes of Colorado and whereas the tribal homelands and seasonal encampments of the Arapoho and Shamb people along the banks of the Cherry Creek and South Plot River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes in the City and County of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas on October 3rd 2016 the City and County of Denver unanimously passed Council Bill 801 series of 2016 officially designating the second Monday of October of each year as Indigenous People's Day in Denver, Colorado. And whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to community, made to the community through Indigenous People's Knowledge, Science, Philosophy, Arts and Culture and through these contributions the City of Denver has developed and thrived. Whereas the Indigenous community especially Yut have made great efforts this year to draw attention to the contributions of Indigenous people including Confluence Week, drawing record of tennis to a National Indigenous Youth Leadership Conference leading conversations on inclusion with their peers and supporting increased Indigenous youth participation in science and engineering. Now therefore be it proclaimed by the Council of the City and County of Denver Section 1 that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history. Our past, present and future and continues to promote the education of the Denver community about these historic and contemporary contributions of Indigenous people. Section 2 at the City and County of Denver, Colorado does hereby observe October 9, 2017 as Indigenous People's Day. Section 3 at the clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation and that a copy be transmitted, excuse me, to the Denver American Indian Commission, the City and County of Denver School District number 1 and the Colorado Commission on Indian Affairs. Thank you Councilman Lopez, your motion to adopt. Mr. President, I move that proclamation number 1127, series of 2017 be adopted. All right, it has been moved and second, it comes with a council councilman Lopez. Thank you Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for the third time but as Indigenous people's day in Denver officially for the second time. It is, it's always awesome to be able to see not just this proclamation come through, come by my desk but to see so many different people from our community in our council chambers. It was a very beautiful piece of artwork that you presented to us earlier and it is exactly the spirit that we drafted this proclamation and this actual, the ordinance that created Indigenous People's Day when we sat down and wrote it and as a community we couldn't think of anything else to begin except for the confluence of the two rivers and those confluence of the two rivers created such a great city and we live in such an amazing city and we we're all proud of it and sometimes we and a lot of people from all over the country are out of the world are proud of it and sometimes a little too proud of it is telling them to go back home. But I'm kidding when I say that but the really nice thing about this is that we are celebrating Indigenous People's Day out of pride for who we are, who we are as a city and the contributions of Indigenous people to the city not out of spite, not out of a replacement of one culture over the other or out of contempt or disrespect. I think of a quote that Sisso Chavez made very very popular and it stuck with me for a very long time and anytime I have the opportunity I speak in front of children and especially children in our community that they often second guests themselves and where they're coming from, who they are and I always say that it's very important to be proud of who you're from and the quote that I use Sisso Chavez is pride in one's own culture does not require contempt or disrespect of another and that's very important. It's very important for us to recognize that no matter who we are where we come from in this society that your pride in your own culture doesn't require the contempt or disrespect of another. Amen, what a year to be for that to just sit on our shoulders for a while for us to think about. And so I wanted to just to thank you all, I think the commission, there's going to be a couple individuals that are going to come speak thank you for your art, your lovely artwork for us to see what's in your heart and what now has become a probably going to be a very important symbol for the community and also just for the work, the daily work every single day we still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners right in poverty without access to services right without access to sobriety or even housing or jobs and what a what a what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores right so we have a lot of work to do and these kind of proclamations in this day is not a day off it's a day on and then right and addressing those those those critical issues so I know that my colleagues are very supportive I'm going to ask you to support this proclamation as I know you always have done in the past I'm very proud of today oh and we made time magazine and newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous peoples day I wanted to make a point out of that thank you uh councilman Lopez and thank you for sponsoring this council martega Mr. President I want to ask that my name be added I don't think I could um add much more to what councilman Lopez has shared with us I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state I worked in the lieutenant governor's office when the commission on Indian affairs was created and had the benefit of being able to go down to the four corners for a peace treaty signing ceremony between the uts and the command cheese that had been sort of at odds with each other for about a hundred years and just being able to participate in that powwow was was pretty awesome so um and for those of you who continue to participate in the annual powwow it's it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture I mean to see that the dance continues to be carried on as well as as the native language from generation to generation is just so incredible because in so many cultures you know people have come here and um assimilated to the you know the norms here and they lose their language and and lose a lot of the culture and in the native community that that hasn't happened that has that you know commitment to just passing that on from generation to generation is is so important and so I'm happy to be a co-sponsor of this tonight thank you all right thank you councilman or take a councilwoman can you thank you very much and I also want to thank my colleague for bringing this forward and I I just wanted to to say a word to the artist about how beautiful and moving I thought this logo was and your description of it um and I think one of the things um that is clear is you know the words sometimes don't convey the power of imagery or music or the other pieces that make up culture and so I think the art is so important and when you talked about water I was also thinking about land and I guess I just wanted to say thank you um many of the native American peoples of Colorado have been at the forefront are actually nationally of defending some of the the um public lands that have been protected over the last few years that are under attack right now and their places that you uh the communities have fought to protect but that everyone gets to enjoy and so I just think that it's an example of where cultural preservation intersects with environmental protection with you know recreation and all of the other ways that that public lands are so important and so I think um I just wanted to say thank you for that because I think we have some very sacred places in our country that are at risk right now and um so as we celebrate I appreciate that there's still a piece of resistance in here and I think that um I just want to mention a solidarity and I mentioned a feeling of solidarity with that resistance so thank you and uh happy confluence week thank you uh councilman can each uh and see you know other comments I'll just say a couple and um in a time of such divisive um ugliness and just despicable behavior from our leadership um the reason I'm so supportive of indigenous peoples days because it means inclusivity it means respecting all respecting those who have been silenced um uh on purpose for a long time and whose history has not been told and so we celebrate inclusivity in the face of such um such evil types honestly

# STEP 2: Analyze & Report

In [11]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [12]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [13]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 02 Aug 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the r

In [14]:
response = tokenizer.decode(outputs[0])

In [15]:
display(Markdown(response))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 02 Aug 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous peoples day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th, please rise with the pledge of allegiance by Councilman Lopez. One makes me under God in the visible liberty of the justice for all. All right. Thank you Councilman Lopez. Madam Secretary, roll call. Black, clerk, espinosa, here, Flynn, Gilmore, here, here, Cashman, here, can each, here, Lopez, new, here, or Teiga, here, Sussan, Mr. President, here, 11 present, 11 members present, we do have a quorum. Approval of the minutes, are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd, stand approved council announcement. So are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just want to do invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock PM. It will move along Broadway from 3rd to Alameda. It's going to be a fun family friendly event. Everyone's invited to come down. Where costume? There will be candy for the kids and there are teaky zombies and 29 herses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right. Thank you, Councilman Clark. We'll be there. All right. Presentations. Madam Secretary, we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening, proclamation 1127, an observance of the annual Indigenous People's Day in the City and County of Denver. Councilman Loplayes, would you please read it? Thank you, Mr. President. We're fried. Proclamation number 17, well let me just say this proclamation number 1127 series of 2017 and observance of the second annual Indigenous People's Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the Indigenous peoples have lived and flourished on the lands known as the Ametica since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes which include the southern Newt, the Yut Mountain, Yut tribes of Colorado and whereas the tribal homelands and seasonal encampments of the Arapoho and Shamb people along the banks of the Cherry Creek and South Plot River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes in the City and County of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas on October 3rd 2016 the City and County of Denver unanimously passed Council Bill 801 series of 2016 officially designating the second Monday of October of each year as Indigenous People's Day in Denver, Colorado. And whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to community, made to the community through Indigenous People's Knowledge, Science, Philosophy, Arts and Culture and through these contributions the City of Denver has developed and thrived. Whereas the Indigenous community especially Yut have made great efforts this year to draw attention to the contributions of Indigenous people including Confluence Week, drawing record of tennis to a National Indigenous Youth Leadership Conference leading conversations on inclusion with their peers and supporting increased Indigenous youth participation in science and engineering. Now therefore be it proclaimed by the Council of the City and County of Denver Section 1 that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history. Our past, present and future and continues to promote the education of the Denver community about these historic and contemporary contributions of Indigenous people. Section 2 at the City and County of Denver, Colorado does hereby observe October 9, 2017 as Indigenous People's Day. Section 3 at the clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation and that a copy be transmitted, excuse me, to the Denver American Indian Commission, the City and County of Denver School District number 1 and the Colorado Commission on Indian Affairs. Thank you Councilman Lopez, your motion to adopt. Mr. President, I move that proclamation number 1127, series of 2017 be adopted. All right, it has been moved and second, it comes with a council councilman Lopez. Thank you Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for the third time but as Indigenous people's day in Denver officially for the second time. It is, it's always awesome to be able to see not just this proclamation come through, come by my desk but to see so many different people from our community in our council chambers. It was a very beautiful piece of artwork that you presented to us earlier and it is exactly the spirit that we drafted this proclamation and this actual, the ordinance that created Indigenous People's Day when we sat down and wrote it and as a community we couldn't think of anything else to begin except for the confluence of the two rivers and those confluence of the two rivers created such a great city and we live in such an amazing city and we we're all proud of it and sometimes we and a lot of people from all over the country are out of the world are proud of it and sometimes a little too proud of it is telling them to go back home. But I'm kidding when I say that but the really nice thing about this is that we are celebrating Indigenous People's Day out of pride for who we are, who we are as a city and the contributions of Indigenous people to the city not out of spite, not out of a replacement of one culture over the other or out of contempt or disrespect. I think of a quote that Sisso Chavez made very very popular and it stuck with me for a very long time and anytime I have the opportunity I speak in front of children and especially children in our community that they often second guests themselves and where they're coming from, who they are and I always say that it's very important to be proud of who you're from and the quote that I use Sisso Chavez is pride in one's own culture does not require contempt or disrespect of another and that's very important. It's very important for us to recognize that no matter who we are where we come from in this society that your pride in your own culture doesn't require the contempt or disrespect of another. Amen, what a year to be for that to just sit on our shoulders for a while for us to think about. And so I wanted to just to thank you all, I think the commission, there's going to be a couple individuals that are going to come speak thank you for your art, your lovely artwork for us to see what's in your heart and what now has become a probably going to be a very important symbol for the community and also just for the work, the daily work every single day we still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners right in poverty without access to services right without access to sobriety or even housing or jobs and what a what a what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores right so we have a lot of work to do and these kind of proclamations in this day is not a day off it's a day on and then right and addressing those those those critical issues so I know that my colleagues are very supportive I'm going to ask you to support this proclamation as I know you always have done in the past I'm very proud of today oh and we made time magazine and newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous peoples day I wanted to make a point out of that thank you uh councilman Lopez and thank you for sponsoring this council martega Mr. President I want to ask that my name be added I don't think I could um add much more to what councilman Lopez has shared with us I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state I worked in the lieutenant governor's office when the commission on Indian affairs was created and had the benefit of being able to go down to the four corners for a peace treaty signing ceremony between the uts and the command cheese that had been sort of at odds with each other for about a hundred years and just being able to participate in that powwow was was pretty awesome so um and for those of you who continue to participate in the annual powwow it's it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture I mean to see that the dance continues to be carried on as well as as the native language from generation to generation is just so incredible because in so many cultures you know people have come here and um assimilated to the you know the norms here and they lose their language and and lose a lot of the culture and in the native community that that hasn't happened that has that you know commitment to just passing that on from generation to generation is is so important and so I'm happy to be a co-sponsor of this tonight thank you all right thank you councilman or take a councilwoman can you thank you very much and I also want to thank my colleague for bringing this forward and I I just wanted to to say a word to the artist about how beautiful and moving I thought this logo was and your description of it um and I think one of the things um that is clear is you know the words sometimes don't convey the power of imagery or music or the other pieces that make up culture and so I think the art is so important and when you talked about water I was also thinking about land and I guess I just wanted to say thank you um many of the native American peoples of Colorado have been at the forefront are actually nationally of defending some of the the um public lands that have been protected over the last few years that are under attack right now and their places that you uh the communities have fought to protect but that everyone gets to enjoy and so I just think that it's an example of where cultural preservation intersects with environmental protection with you know recreation and all of the other ways that that public lands are so important and so I think um I just wanted to say thank you for that because I think we have some very sacred places in our country that are at risk right now and um so as we celebrate I appreciate that there's still a piece of resistance in here and I think that um I just want to mention a solidarity and I mentioned a feeling of solidarity with that resistance so thank you and uh happy confluence week thank you uh councilman can each uh and see you know other comments I'll just say a couple and um in a time of such divisive um ugliness and just despicable behavior from our leadership um the reason I'm so supportive of indigenous peoples days because it means inclusivity it means respecting all respecting those who have been silenced um uh on purpose for a long time and whose history has not been told and so we celebrate inclusivity in the face of such um such evil types honestly<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Summary**
Attendees: Denver City Council members
Location: Denver City Council chambers
Date: Monday, October 9th
Summary: The Denver City Council approved a proclamation recognizing Indigenous People's Day in the City and County of Denver.

**Discussion Points**
* The confluence of two rivers, which inspired the logo and the name "Confluence Week"
* The importance of water and its significance in recent events and politics
* The recognition of Indigenous peoples' contributions to the city's history and present-day contributions
* The celebration of Indigenous People's Day as a way to honor and promote Indigenous culture

**Takeaways**
* The Denver City Council recognizes the importance of Indigenous peoples' contributions to the city's history and present-day contributions
* The city will continue to promote education about Indigenous peoples' contributions and history
* The proclamation acknowledges the cultural and foundational contributions of Indigenous peoples to the city's past, present, and future

**Action Items**
* Councilman Lopez to read the proclamation and explain its significance
* Councilman Lopez to thank the artist for creating the logo and to acknowledge the contributions of the Native American community to the city
* Councilwoman Cashman to express solidarity with the resistance against public land attacks
* Councilman Clark to invite the public to attend the first-ever Halloween parade on Broadway in Lucky District 7
* Councilman Lopez to sponsor the proclamation and to express his pride in celebrating Indigenous People's Day

**Action Items with Owners**
* Councilman Lopez to sponsor the proclamation (done)
* Councilwoman Cashman to express solidarity with the resistance against public land attacks (done)
* Councilman Clark to invite the public to attend the Halloween parade (done)
* Councilman Lopez to thank the artist for creating the logo (done)<|eot_id|>

# Student contribution

Student MS By research. has made this powerful variation that uses `TextIteratorStreamer` to stream back results into a Gradio UI, and takes advantage of background threads for performance! I'm sharing it here if you'd like to take a look at some very interesting work. Thank you, Kid!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D